# Rapid specification of regression workflows

### *Math::Regressionizer demo*


Anton Antonov    
[RakuForPrediction at WordPress](https://pythonforprediction.wordpress.com)
October 2026


## Introduction

This document provides examples of specifying different regression workflows
using the class `Math::Regressionizer` of the Raku package ["Math::Regressionizer"](https://github.com/antononcube/Raku-Math-Regressionizer), [AAp1].

The primary focus of `Math::Regressionizer` is [Quantile Regression (QR)](https://en.wikipedia.org/wiki/Quantile_regression), [RK1, RK2].
It closely follows the monadic pipeline design explained in detail in the document
["A monad for Quantile Regression workflows"](https://github.com/antononcube/MathematicaForPrediction/blob/master/MarkdownDocuments/A-monad-for-Quantile-Regression-workflows.md), [AA1].

For introduction and overview of Quantile Regression see the video
["Boston useR! QuantileRegression Workflows 2019-04-18"](https://www.youtube.com/watch?v=a_Dk25xarvE).

### Summary of `Math::Regressionizer` features

- The class `Math::Regressionizer` facilitates rapid specifications of regressions workflows.
  - To quickly specify:
    - data rescaling and summary
    - regression computations
    - outliers finding
    - conditional Cumulative Distribution Functions (CDFs) reconstruction
    - plotting of data, fits, residual errors, outliers, CDFs

- `Math::Regressionizer` works with data frames, numpy arrays, lists of numbers, and lists of numeric pairs.

### Details and arguments

- The curves computed with Quantile Regression are called **regression quantiles**.

- `Math::Regressionizer` has three regression methods:
  - `quantile-regression`
  - `quantile-regression-fit`
  - `least_squares_fit`

- The regression quantiles computed with the methods `quantile-regression` and `quantile-regression-fit`
  correspond to probabilities specified with the argument `:probs(:@probabilities)`.

- The method`quantile-regression` computes fits using a B-spline functions basis.
  - The basis is specified with the arguments `:@knots` and `:order(:$interpolation-order)`.
  - `:order(:$interpolation-order)` is 3 by default.

- The methods `quantile-regession-fit` and `least-squares-fit` use lists of basis functions to fit with
  specified with the argument `:funcs(:@basis)`.

### Workflows flowchart

The following flowchart summarizes the workflows that are supported by `Math::Regressionizer`:

![](https://raw.githubusercontent.com/antononcube/Python-Regressionizer/main/docs/img/Quantile-regression-workflow-extended.jpg)


### Previous work

Roger Koenker implemented the R package "quantreg", [RKp1].
Anton Antonov implemented the R package "QRMon-R" for the specification of monadic pipelines for doing QR, [AAp1].

Several Wolfram Language (aka Mathematica) packages are implemented by Anton Antonov, see [AAp1, AAp2, AAf1].

**Remark:** The paclets at the Wolfram Language Paclet Repository were initially Mathematica packages hosted at GitHub.
The Wolfram Function Repository function
[`QuantileRegression`](https://resources.wolframcloud.com/FunctionRepository/resources/QuantileRegression/), [AAf1]
does only B-spline fitting.

-------

## Setup

Load the "Math::Regressionizer" and other data science packages:

In [1]:
use Math::Regressionizer;
use Math::Polynomial::Chebyshev;

use Data::Importers;
use Data::TypeSystem;

use Statistics::Distributions;
use Stats;

use JavaScript::Google::Charts;

In [ ]:
my $backgroundColor = '#1A1A1A';
my $format = 'html'; 
my $titleTextStyle = %(color => 'DarkGray', fontSize => 16);
my $legend = {
    position => 'right',
    textStyle => {
        color => 'DarkGray',
        fontSize => 12
    }
};
my $chartArea = {
    :$backgroundColor,
    left => 50,
    right => 100,
    top => 50, 
    bottom => 50
}

my $data-color = 'DarkGray';
my %dark-opts = :$backgroundColor, :$titleTextStyle, :$legend, :$chartArea;

# Offset from 1900-01-01 to 1970-01-01
my $date-offset = 2208988800;

-------

## Generate input data

Generate random data:

In [3]:
my @x-values = -3, -2.99 ... 3.01;
my @dist-data = @x-values.map({
    [$_, exp( - $_ ** 2) + random-variate(NormalDistribution.new(0, 0.15 * sqrt(abs(1.5 - $_) / 1.5)))]
});

deduce-type(@dist-data)

Vector((Any), 602)

Plot the generated data:

In [ ]:
#% html
js-google-charts('Scatter', 
    @dist-data,
    title => 'Distribution data',
    :$titleTextStyle,
    series => { 0 => {color => 'SteelBlue', pointSize => 2} },
    :$backgroundColor, 
    :$chartArea, 
    :$format,
    :700width,
    :450height,
    div-id => 'dist-data')

------

## Fit given functions

Define a list of functions:

In [5]:
my @funcs = {1}, {$_}, {cos($_)}, {cos(3 * $_)}, {cos(6 * $_)};
@funcs.elems

5

In [6]:
my @chebyshev-polynomials = (^10).map(-> $order { -> $x { chebyshev-t($order, $x) } });
@chebyshev-polynomials.elems


10

Define ***regression quantile*** probabilities:

In [7]:
my @probs = 0.1, 0.5, 0.9;

[0.1 0.5 0.9]

Perform Quantile Regression and (non-linear) Least Squares Fit:

In [8]:
my $obj = 
    Math::Regressionizer.new(@dist-data)
    .echo-data-summary()
    .quantile-regression-fit(@chebyshev-polynomials, :@probs)
    .least-squares-fit(@chebyshev-polynomials)
    .plot(
        title => "Quantile Regression and Least Squares fitting using Chebyshev polynomials", 
        :$titleTextStyle,
        :$backgroundColor, 
        :$legend
    )

Statistic    regressor | value
min	-3 | -0.7186904207058399
25%	-1.4975 | 0.017687099071416253
median	0.005 | 0.17513205492223055
75%	1.5075 | 0.5968836365361779
max	3.01 | 1.2452573179257929


Math::Regressionizer( data => 602 points, basis-functions => 10 count, regression-quantiles => at (0.1, 0.5, 0.9, mean), value => Str)

Plot the obtained regression quantilies and least squares fit:

In [ ]:
#% html
$obj.take-value

-------

## Fit B-splines

Instead of coming-up with basis functions we can use B-spline basis:

In [10]:
my $obj = 
    Math::Regressionizer.new(@dist-data)
    .quantile-regression(:8knots, probs => [0.2, 0.5, 0.8])
    .plot(title => "B-splines fit", |%dark-opts, div-id => 'b-splines')

Math::Regressionizer( data => 602 points, basis-functions => 0 count, regression-quantiles => at (0.2, 0.5, 0.8), value => Str)

Show the obtained plot:

In [ ]:
#% html
$obj.take-value

Here is a hashmap (dictionary) of the found regression quantiles:

In [12]:
.say for $obj.take-regression-quantiles

0.8 => -> +@args { #`(Block|4562322242448) ... }
0.5 => -> +@args { #`(Block|4562322221464) ... }
0.2 => -> +@args { #`(Block|4562305947624) ... }


------

## Weather temperature data

Load weather data:

In [13]:
my $url = "https://raw.githubusercontent.com/antononcube/MathematicaVsR/master/Data/MathematicaVsR-Data-Atlanta-GA-USA-Temperature.csv";
my @dsTemperature = data-import($url, headers => 'auto');
@dsTemperature .= deepmap({ $_ ~~ / ^ \d+ '-' / ?? Date.new($_) !! $_.Num });
@dsTemperature .= grep( 2020 ≤ *<Date>.year ≤ 2023);
deduce-type(@dsTemperature)

Vector(Struct([AbsoluteTime, Date, Temperature], [Num, Date, Num]), 1461)

Convert to "numpy" array: 

In [14]:
my @temp-data = @dsTemperature.map(*<AbsoluteTime Temperature>);
deduce-type(@temp-data)

Vector(Vector(Atom((Numeric)), 2), 1461)

Here is pipeline for Quantile Regression computation and making of a corresponding plot:

In [15]:
my $obj =
    Math::Regressionizer.new(@temp-data)
    .echo-data-summary()
    .quantile-regression(knots => 20, probs => [0.2, 0.5, 0.8])
    .date-list-plot(
        :$date-offset,
        title => "Atlanta, Georgia, USA, Temperature, ℃", 
        |%dark-opts,
        width => 1200,
        div-id => 'temp-data-fits'
    )

Statistic    regressor | value
min	3786825600 | -11.89
25%	3818361600 | 10.06
median	3849897600 | 16.94
75%	3881433600 | 22.56
max	3912969600 | 32.39


Math::Regressionizer( data => 1461 points, basis-functions => 0 count, regression-quantiles => at (0.2, 0.5, 0.8), value => Str)

Show the obtained plot:

In [ ]:
#% html
$obj.take-value

Here we show the fractions of the number of points under each regression quantile:

In [17]:
$obj.separate(:cumulative, :fractions).take-value

{0.2 => 0.201232, 0.5 => 0.50308, 0.8 => 0.80219}

**Remark:** If the quantile regression algorithms work correctly then the cumulation separation fractions correspond -- i.e. are nearly equal - to the probabilities of the regression quantiles.

**Remark:** The usage messages of the arguments of `Math::Regressionizer`'s methods can provide additional details or guidance. See those messages with this code:

```raku
.say for Math::Regressionizer.^lookup('plot').signature.params.map({ $_.name => $_.WHY })
```

-------

## Fitting errors

### Errors

Here the absolute fitting errors are computed and the average is for each is computed:

In [18]:
$obj.errors(:relative-errors).take-value.map({ $_.key => mean($_.value».tail) })

(0.5 => -0.29080928930005573 0.2 => 0.13704748372808573 0.8 => -0.7711321574537081)

### Error plots

Here we give the fitting errors (residuals) for the regression quantiles found and plotted above:

In [19]:
$obj .= errors

Math::Regressionizer( data => 1461 points, basis-functions => 0 count, regression-quantiles => at (0.2, 0.5, 0.8), value => Hash)

In [ ]:
#% html
$obj.take-value.sort(*.key).map({
    js-google-charts('Scatter', $_.value, title => "Errors of regression quantile at {$_.key}", |%dark-opts, :$format, div-id => "errors-{ $_.head }") }).join("\n\n")

------

## Outliers

One way to find _contextual_ outliers in time series is to find regression quantiles at low- and high enough probabilities, and then select the points "outside" of those curves:

In [21]:
my $obj = 
    Math::Regressionizer.new(@temp-data)
    .quantile-regression(:20knots, probs => [0.01, 0.99], :3order)
    .outliers;

$obj.take-value

{bottom => [[3788553600 -3.11] [3795552000 7.39] [3797798400 9.28] [3798230400 10.22] [3806870400 20.11] [3809721600 12.39] [3822508800 -4.72] [3833395200 18.72] [3845836800 3.5] [3888259200 3.89] [3889987200 7.06] [3900355200 21.72]], top => [[3790540800 20.39] [3794428800 22.39] [3814128000 21.28] [3820608000 17.78] [3849984000 21.06] [3855600000 20.5] [3857673600 22.39] [3864153600 29.83] [3864240000 29.67] [3872707200 25.61] [3879446400 18.67]]}

Here we plot the outliers (using a "narrower band" than above):

In [ ]:
my $obj = 
    Math::Regressionizer.new(@temp-data)
    .quantile-regression(:20knots, probs => [0.05,  0.95], :3order)
    .outliers-plot(
        title => "Outliers of Atlanta, Georgia, USA, Temperature, ℃",
        :date-plot, 
        |%dark-opts,
        :$data-color,
        :1200width, :400height
    );

$obj.take-value

---------

## Pick points along regression quantiles  

Here is a workflow that finds a regression quantile for probability `0.5` and pick data points around it:

In [59]:
my @probs = 0.5;
my $obj=
    Math::Regressionizer(@temp-data)
    .quantile-regression(:20knots, :@probs)
    .pick-path-points(threshold => 1.5, :!relative-errors)
    ;

my %path-points = $obj.take-value{@probs[0]}.flat(:hammer);
deduce-type(%path-points)

Assoc(Atom((Str)), Atom((Numeric)), 491)

Plot the data and the path points:

In [ ]:
#% html
my @wide-data = $obj.take-data.map({ 
    %(
        x => DateTime.new(Instant.from-posix($_.head)),
        data => $_.tail,
        path => %path-points{$_.head} // Nil
    )
});

js-google-charts('Scatter', @wide-data, column-names => <x data path>, |%dark-opts, pointSize => 2, :$format, div-id => 'path-points')

--------

## Conditional CDF

Here is a list of probabilities to be used to reconstruct Cumulative Distribution Functions (CDFs):

In [ ]:
my @probs = 0.01, |(0.1, 0.3 ... 0.9), 0.99;

[0.01 0.1 0.3 0.5 0.7 0.9 0.99]

Here we find the regression quantiles for those probabilities:

In [58]:
my $obj =
    Math::Regressionizer(@temp-data)
    .quantile-regression(:20knots, :@probs)
    .date-list-plot(|%dark-opts, data-color => 'SteelBlue', :1200width, :$format, div-id => 'for-CDF')

Math::Regressionizer( data => 1461 points, basis-functions => 0 count, regression-quantiles => at (0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99), value => Str)

Here we show the plot obtained above:

In [ ]:
#% html
$obj.take-value

### Get CDF function

Here we take a date in ISO format and convert to number of seconds since 1900-01-01:

In [57]:
my $iso-date = "2022-01-01";
my $date-object = DateTime.($iso-date);
my $epoch = DateTime.new(year => 1900, month => 1, day => 1);

my $focusPoint = ($date-object - $epoch);

3849984027

Here the _conditional_ CDF at that date is computed:

In [56]:
my %CDFs = $obj.conditional-cdf($focusPoint).take-value;

{3849984027 => sub { }}

Plot the obtained CDF function:

In [ ]:
#% html
my @span = $obj.take-regression-quantiles{0.01}($focusPoint), $obj.take-regression-quantiles{0.99}($focusPoint);
@xs = @span[0], @span[0] + ((@span[*-1] - @span[0])/20) ... @span[*-1];

my @cdf-values = @xs.map({ [$_, %CDFs{$focusPoint}($_)] });

js-google-charts('Line', 
    @cdf-values, 
    title => "Temperature Data CDF at $focusPoint ({Date.new(Instant.from-posix($focusPoint - $date-offset))})",
    :$titleTextStyle,
    :$backgroundColor,
    :$chartArea,
    :$format,
    hAxis => {title => 'Temperature', titleTextStyle => {color => 'Gray'}},
    vAxis => {title => 'Probability', titleTextStyle => {color => 'Gray'}},
    div-id => 'CDF'
)

### Plot multiple CDFs

Here are few dates converted into number of seconds since 1990-01-01:

In [52]:
my @pointsForCDFs = $focusPoint <<+>> (-1 ... 2) <<*>> (365 * 24 * 3600);

[3818448027 3849984027 3881520027 3913056027]

Here are the plots of CDF at those dates:

In [ ]:
#% html
$obj.conditional-cdf-plot(@pointsForCDFs, title => 'CDFs', |%dark-opts).take-value

-----

## Simulation

Here is a pipeline that produces simulated time series based on fitted regression quantiles:

In [ ]:
my @sim-points =
    Math::Regressionizer.new(@temp-data)
    .echo-data-summary()
    .quantile-regression(:20knots, probs => [0.001, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.999])
    .simulate(800)
    .take-value;

deduce-type(@sim-points)

Here is the plot of the original and simulated time series:

In [ ]:
# TBD...

------

## References

### Articles, books

[RK1] Roger Koenker, 
[Quantile Regression](https://books.google.com/books/about/Quantile_Regression.html?id=hdkt7V4NXsgC), 
Cambridge University Press, 2005.

[RK2] Roger Koenker,
["Quantile Regression in R: a vignette"](https://cran.r-project.org/web/packages/quantreg/vignettes/rq.pdf),
(2006),
[CRAN](https://cran.r-project.org/).

[AA1] Anton Antonov,
["A monad for Quantile Regression workflows"](https://github.com/antononcube/MathematicaForPrediction/blob/master/MarkdownDocuments/A-monad-for-Quantile-Regression-workflows.md),
(2018),
[MathematicaForPrediction at GitHub](https://github.com/antononcube/MathematicaForPrediction).

### Packages, paclets

[AAp1] Anton Antonov,
[Quantile Regression Python package](https://github.com/antononcube/Python-Regressionizer),
(2024),
[GitHub/antononcube](https://github.com/antononcube).

[AAp2] Anton Antonov,
[`QRMon-R`](https://github.com/antononcube/QRMon-R),
(2019),
[GitHub/antononcube](https://github.com/antononcube).

[AAp3] Anton Antonov,
[Quantile Regression WL paclet](https://github.com/antononcube/WL-QuantileRegression-paclet),
(2014-2023),
[GitHub/antononcube](https://github.com/antononcube).

[AAp4] Anton Antonov,
[Monadic Quantile Regression WL paclet](https://github.com/antononcube/WL-MonadicQuantileRegression-paclet),
(2018-2024),
[GitHub/antononcube](https://github.com/antononcube).

[AAf1] Anton Antonov,
[`QuantileRegression`](https://resources.wolframcloud.com/FunctionRepository/resources/QuantileRegression),
(2019),
[Wolfram Function Repository](https://resources.wolframcloud.com/FunctionRepository/resources/QuantileRegression).

[RKp1] Roger Koenker,
[`quantreg`](https://cran.r-project.org/web/packages/quantreg/index.html),
[CRAN](https://cran.r-project.org/).

### Repositories

[AAr1] Anton Antonov,
[DSL::English::QuantileRegressionWorkflows in Raku](https://github.com/antononcube/Raku-DSL-English-QuantileRegressionWorkflows),
(2020),
[GitHub/antononcube](https://github.com/antononcube/Raku-DSL-English-QuantileRegressionWorkflows).


### Videos

[AAv1] Anton Antonov,
["Boston useR! QuantileRegression Workflows 2019-04-18"](https://www.youtube.com/watch?v=a_Dk25xarvE),
(2019),
[Anton Antonov at YouTube](https://www.youtube.com/@AAA4Prediction).

[AAv2] Anton Antonov,
["useR! 2020: How to simplify Machine Learning workflows specifications"](https://www.youtube.com/watch?v=b9Uu7gRF5KY),
(2020),
[R Consortium at YouTube](https://www.youtube.com/channel/UC_R5smHVXRYGhZYDJsnXTwg).